In [17]:
import pandas as pd
import re

# 합계출산율(모의 연령별출산율) 불러오기 -> 확인하기
df = pd.read_csv("시도_합계출산율__모의_연령별_출산율_20260520112924.csv", header=[0, 1])
df.head()

시도별   1993                                                             \
     시도별  합계출산율 모의 연령별출산율:15-19세 20-24세 25-29세 30-34세 35-39세 40-44세 45-49세   
0     전국  1.654              4.4   71.9  176.5   63.2   13.5    2.0    0.2   
1  서울특별시  1.558              2.7   54.2  168.4   70.0   14.0    1.8    0.2   
2  부산광역시  1.505              2.9   58.6  167.3   58.2   11.9    1.7    0.1   
3  대구광역시  1.556              2.8   62.1  174.7   58.6   10.9    1.7    0.2   
4  인천광역시  1.815              6.1   96.4  182.9   62.1   12.8    2.1    0.1   

    1994  ...   2023          2024                                        \
   합계출산율  ... 40-44세 45-49세  합계출산율 모의 연령별출산율:15-19세 20-24세 25-29세 30-34세   
0  1.656  ...    7.9    0.2  0.748              0.4    3.8   20.7   70.4   
1  1.565  ...    8.7    0.2  0.581              0.2    1.3    8.3   49.4   
2  1.468  ...    7.5    0.2  0.683              0.2    2.7   16.5   65.5   
3  1.557  ...    6.9    0.2  0.754              0.3    3.2   19.7   74.5   
4  1.776  ...    7.8    0.1  0.762              0.4    4.6   23.2   70.4   

                        
  35-39세 40-44세 45-49세  
0   46.0    7.7    0.2  
1   46.9    8.9    0.2  
2   44.0    6.8    0.3  
3   45.0    7.0    0.2  
4   45.9    7.8    0.2  

[5 rows x 257 columns]

In [18]:
# 데이터 전처리 

# 합계출산율 열만 선택
합계출산율 = df.loc[:, df.columns.get_level_values(1) == "합계출산율"]

# 컬럼명을 연도만 남기기(합계출산율 컬럼명 제거)
합계출산율.columns = 합계출산율.columns.get_level_values(0)

# 시도별을 인덱스로 지정
합계출산율.index = df[("시도별", "시도별")]

# 인덱스 이름 "(시도별, 시도별)" -> "시도별"
합계출산율.index.name = "시도별"

# reset_index()는 인덱스를 초기화하면서 기존 인덱스를 컬럼으로 만듦
합계출산율 = 합계출산율.reset_index().melt(
    id_vars="시도별",
    var_name="연도",
    value_name="합계출산율"
)

합계출산율["연도"] = pd.to_numeric(합계출산율["연도"])
합계출산율 = 합계출산율.loc[(합계출산율["연도"] >= 2000) & (합계출산율["연도"] <= 2021), :]

합계출산율


,시도별,연도,합계출산율
126,전국,2000,1.480
127,서울특별시,2000,1.275
128,부산광역시,2000,1.235
129,대구광역시,2000,1.378
130,인천광역시,2000,1.473
...,...,...,...
517,전북특별자치도,2021,0.85
518,전라남도,2021,1.017
519,경상북도,2021,0.966
520,경상남도,2021,0.903


In [19]:
# 1인당 지역내총생산, 1인당 민간소비, 1인당 지역총소득, 1인당 개인소득 데이터 불러오기 -> 확인하기
df = pd.read_csv("시도별_1인당_지역내총생산__지역총소득__개인소득_20260520112555.csv", header=[0, 1])
df.head()

시도별       1985       1986       1987       1988       1989       1990  \
     시도별 1인당 지역내총생산 1인당 지역내총생산 1인당 지역내총생산 1인당 지역내총생산 1인당 지역내총생산 1인당 지역내총생산   
0     전국       2245       2597       2994       3548       3971       4794   
1  서울특별시       2377       2779       3230       3701       4185       5086   
2  부산광역시       1932       2174       2495       2829       3036       3787   
3  대구광역시       1850       2166       2546       2845       3121       3819   
4  인천광역시       2743       3090       3484       4131       4559       5422   

        1991       1992       1993  ...     2020                2021  \
  1인당 지역내총생산 1인당 지역내총생산 1인당 지역내총생산  ... 1인당 개인소득 1인당 민간소비 1인당 지역내총생산   
0       5757       6519       7299  ...    21342    17320      40271   
1       6185       7049       8104  ...    24226    21185      49680   
2       4470       4795       5294  ...    20460    17365      29395   
3       4332       4725       5081  ...    20229    17276      25543   
4       6645       7066       7609  ...    20310    16059      33551   

                                 2022 p)                              
  1인당 지역총소득 1인당 개인소득 1인당 민간소비 1인당 지역내총생산 1인당 지역총소득 1인당 개인소득 1인당 민간소비  
0     40723    22059    18431      41948     42563    23388    20078  
1     54571    24950    22603      51612     57236    26112    24455  
2     31414    21035    18712      31611     32293    22577    20637  
3     29552    20886    18362      26736     31056    22368    19903  
4     35499    21170    17176      35295     37442    22406    18706  

[5 rows x 113 columns]

In [20]:
# 데이터 전처리 -> 1인당 지역총소득 분리

# p(provisional). 잠정치 제거
경제 = df.drop(columns="2022 p)", level=0)

# 1인당 지역내총생산, 1인당 민간소비 제거 -> 1인당 지역총소득, 1인당 개인소득만 남김
경제.drop(columns=["1인당 지역내총생산", "1인당 민간소비"], level=1, inplace=True)

# 1인당 지역총소득 컬럼만 분리
일인당지역총소득 = 경제.loc[:, 경제.columns.get_level_values(1) == "1인당 지역총소득"]

# 일인당지역총소득 컬럼명 버리기
일인당지역총소득.columns = 일인당지역총소득.columns.droplevel(1)

# 시도별을 인덱스로 지정
# 일인당지역총소득.index = df[("시도별", "시도별")]
# 일인당지역총소득.index.name = "시도별"

일인당지역총소득.insert(0, "시도별", df[("시도별", "시도별")])

일인당지역총소득 = 일인당지역총소득.melt(
    id_vars="시도별",
    var_name="연도",
    value_name="1인당 지역총소득"
)

일인당지역총소득

,시도별,연도,1인당 지역총소득
0,전국,2000,13860
1,서울특별시,2000,17194
2,부산광역시,2000,10933
3,대구광역시,2000,10684
4,인천광역시,2000,10633
...,...,...,...
391,전라북도,2021,31803
392,전라남도,2021,37741
393,경상북도,2021,38071
394,경상남도,2021,33363


In [21]:
# 데이터 전처리 -> 1인당 개인소득 분리

일인당개인소득 = 경제.loc[:, 경제.columns.get_level_values(1) == "1인당 개인소득"]

# "1인당 개인소득" 컬럼명 버리기
일인당개인소득.columns = 일인당개인소득.columns.droplevel(1)

# 시도별 컬럼 넣기
일인당개인소득.insert(
    0,
    "시도별",
    df[("시도별", "시도별")]
)

# wide -> long
일인당개인소득 = 일인당개인소득.melt(
    id_vars="시도별",
    var_name="연도",
    value_name="1인당 개인소득"
)

일인당개인소득

,시도별,연도,1인당 개인소득
0,전국,2000,8694
1,서울특별시,2000,9978
2,부산광역시,2000,8296
3,대구광역시,2000,8274
4,인천광역시,2000,7651
...,...,...,...
391,전라북도,2021,20745
392,전라남도,2021,20779
393,경상북도,2021,20522
394,경상남도,2021,20535


In [22]:
# 고용률 데이터 불러오기 -> 확인
df = pd.read_csv("고용률_시도__20260520115827.csv")

# 월별로 되어있음. 연도별로 바꾸고 평균 내야할 듯
df.head()

,시도별,성별,1999.06,1999.07,1999.08,1999.09,1999.10,1999.11,1999.12,2000.01,...,2025.07,2025.08,2025.09,2025.10,2025.11,2025.12,2026.01,2026.02,2026.03,2026.04
0,계,계,57.6,57.5,57.3,58.6,59.0,58.8,57.5,56.2,...,63.4,63.3,63.7,63.4,63.4,61.5,61.0,61.8,62.7,63.0
1,계,남자,69.8,69.9,69.8,70.8,71.2,71.2,70.2,68.9,...,71.0,71.0,71.1,70.8,70.9,69.8,68.9,69.3,70.2,70.5
2,계,여자,46.2,46.0,45.6,47.1,47.6,47.2,45.6,44.3,...,56.0,55.8,56.4,56.2,56.0,53.4,53.2,54.6,55.3,55.7
3,서울특별시,계,56.8,56.7,56.5,57.5,57.8,58.4,58.1,57.5,...,62.4,62.3,62.2,61.9,61.8,60.9,60.1,60.8,61.3,61.4
4,서울특별시,남자,68.3,68.1,68.5,69.2,69.7,70.6,70.0,69.6,...,68.9,69.2,68.3,67.9,68.6,67.5,66.2,66.6,67.3,67.2


In [23]:
# 데이터 전처리

# 월별 컬럼 찾기 -> 1999.01
month_cols = [
    col for col in df.columns
    if re.match(r"^\d{4}\.\d{2}$", str(col))
]

# 연도 목록 만들기 -> 1999, 2000
years = sorted(set(col[:4] for col in month_cols))

# 시도별, 성별 컬럼은 그대로 두기
고용률 = df[["시도별", "성별"]].copy()

# 같은 연도에 해당하는 12개월 컬럼들의 평균 계산
for year in years:
    # 특정 연도에 해당하는 월별 컬럼들만 골라내기
    year_cols = [col for col in month_cols if col.startswith(year)]

    고용률[year] = df[year_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)


# 성별 데이터 버리고 합계로
고용률 = 고용률.loc[고용률["성별"] == "계", :]

# 합계 데이터만 있으므로, 성별 컬럼도 버림
고용률.drop(columns="성별", inplace=True)

# 시도별이 "계"라고 되어있는걸 "전국"으로 통일
고용률.loc[0, "시도별"] = "전국"

# wide -> long
고용률 = 고용률.melt(
    id_vars="시도별",
    var_name="연도",
    value_name="고용률" 
)

# 정수형으로
고용률["연도"] = pd.to_numeric(고용률["연도"])

# 2000년 ~ 2021까지의 데이터만
고용률 = 고용률.loc[(고용률["연도"] >= 2000) & (고용률["연도"] <= 2021), :]

고용률

,시도별,연도,고용률
18,전국,2000,58.508333
19,서울특별시,2000,58.250000
20,부산광역시,2000,55.366667
21,대구광역시,2000,55.991667
22,인천광역시,2000,58.175000
...,...,...,...
409,전북특별자치도,2021,61.241667
410,전라남도,2021,64.616667
411,경상북도,2021,61.008333
412,경상남도,2021,60.666667


In [ ]:
# 저장하기

# 한글깨짐방지 : encoding="utf-8-sig"
합계출산율.to_csv("합계출산율.csv", encoding="utf-8-sig")
일인당지역총소득.to_csv("일인당지역총소득.csv", encoding="utf-8-sig")
일인당개인소득.to_csv("일인당개인소득.csv", encoding="utf-8-sig")
고용률.to_csv("고용률.csv", encoding="utf-8-sig")

# result = 합계출산율
# result[""]